# PortWatch AI — Prediction Refresh Notebook (Local Version)

**Converted from:** Azure Databricks (Spark + ADLS)
**Target:** Local Python + Pandas + joblib

### Changes from Original:
- **REMOVED:** `abfss://` paths and ADLS connections.
- **REMOVED:** Spark/DBFS specific operations (`dbutils`, `spark.read`, `spark.createDataFrame`).
- **REMOVED:** Re-reading the 700MB raw dataset (superfluous, maximum date is extracted from features seamlessly).
- **ADDED:** Explicit `feature_columns.txt` checks to strictly guarantee no feature mismatch between the training run and this inference run.
- **FIXED (CRITICAL BUG):** The original notebook simply duplicated the last historical row 7 times to simulate the future, but *failed* to recalculate the calendar variables for those future days. This meant if the last historical day was a Tuesday, the model would predict the entire next week as if every day were a Tuesday! This local version natively recalculates `dow` and `is_weekend` to preserve feature integrity.

In [1]:
# =============================================================================
# Cell 1 — CONFIG & IMPORTS
# =============================================================================
import os
import time
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

# Suppress pandas fragmentation warnings
import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

os.chdir(r'd:\notebooks')

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
OUTPUTS_DIR = BASE_DIR / "outputs"

EXPANDED_FEATURES_PATH = DATA_DIR / "port_daily_expanded.parquet"
LGBM_MODEL_PATH = MODELS_DIR / "portwatch_lightgbm.pkl"
FEATURE_LIST_PATH = MODELS_DIR / "feature_columns.txt"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "latest_predictions.parquet"
VALIDATION_PREDS_PATH = OUTPUTS_DIR / "validation_predictions.parquet"

print("Paths Configured:")
print(f"  Features: {EXPANDED_FEATURES_PATH}")
print(f"  Model:    {LGBM_MODEL_PATH}")

Paths Configured:
  Features: data\port_daily_expanded.parquet
  Model:    models\portwatch_lightgbm.pkl


In [2]:
# =============================================================================
# Cell 2 — LOAD MODEL & METADATA
# =============================================================================
print("Loading artifacts...")

# Load LightGBM Model
model = joblib.load(LGBM_MODEL_PATH)
print(f"\u2705 Model loaded: {type(model).__name__}")

# Load required feature sequence from training
with open(FEATURE_LIST_PATH, 'r') as f:
    TRAIN_FEATURES = f.read().strip().split('\n')

print(f"\u2705 Feature list loaded: {len(TRAIN_FEATURES)} mandatory features recognized.")

Loading artifacts...


✅ Model loaded: Booster
✅ Feature list loaded: 28 mandatory features recognized.


In [3]:
# =============================================================================
# Cell 3 — LOAD EXPANDED HISTORICAL FEATURES
# =============================================================================
pdf_feat = pd.read_parquet(EXPANDED_FEATURES_PATH)
pdf_feat['event_date'] = pd.to_datetime(pdf_feat['event_date'])

last_date = pdf_feat['event_date'].max()
print(f"Loaded expanded features: {len(pdf_feat):,} total rows.")
print(f"Last valid historical feature date: {last_date.date()}")

Loaded expanded features: 125,500 total rows.
Last valid historical feature date: 2025-11-14


In [4]:
# =============================================================================
# Cell 4 — GENERATE 7-DAY FUTURE PROJECTION (WITH CRITICAL BUG FIX)
# =============================================================================
PRED_DAYS = 7
future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=PRED_DAYS,
    freq="D"
)
print(f"Predicting for {PRED_DAYS} future days: {future_dates.min().date()} \u2192 {future_dates.max().date()}")

latest_rows = (
    pdf_feat
    .sort_values("event_date")
    .groupby("portid")
    .tail(1)
    .copy()
)

future_rows = []
for d in future_dates:
    temp = latest_rows.copy()
    temp["event_date"] = d
    
    # ----------------------------------------------------------------------
    # 🚨 CRITICAL BUG FIX: RECOMPUTE CALENDAR FEATURES
    # The original Databricks pipeline naively duplicated the last row 7 times 
    # but permanently preserved the old `dow` and `is_weekend` columns. 
    # We must explicitly recalculate them here using pandas logic to ensure 
    # there's no feature drift between training properties and inference!
    # ----------------------------------------------------------------------
    temp['year'] = d.year
    temp['month'] = d.month
    temp['dow'] = d.dayofweek  # Monday=0, Sunday=6
    temp['is_weekend'] = temp['dow'].isin([5, 6]).astype(int)
    
    future_rows.append(temp)

future_df = pd.concat(future_rows, ignore_index=True)
print(f"Successfully generated future sequence matrix: {len(future_df):,} rows.")

Predicting for 7 future days: 2025-11-15 → 2025-11-21
Successfully generated future sequence matrix: 350 rows.


In [5]:
# =============================================================================
# Cell 5 — EXECUTE LIGHTGBM INFERENCE
# =============================================================================
# Guarantee inference columns perfectly align with the loaded `TRAIN_FEATURES`
missing_cols = set(TRAIN_FEATURES) - set(future_df.columns)
if missing_cols:
    raise ValueError(f"FATAL: Inference dataframe is critically missing features: {missing_cols}")

X_future = future_df[TRAIN_FEATURES].fillna(0.0)

t0 = time.time()
# Predict using the LightGBM optimal iteration
preds = model.predict(X_future, num_iteration=model.best_iteration)

# Clip logic (don't allow negative port calls) & round to 3 decimals
future_df["pred_daily_port_calls"] = np.clip(preds, a_min=0, a_max=None).round(3)

print(f"\u2705 Future predictions computed in {time.time()-t0:.2f}s")
print(future_df[["event_date", "portid", "dow", "is_weekend", "pred_daily_port_calls"]].head(10).to_string())

✅ Future predictions computed in 0.01s
  event_date    portid  dow  is_weekend  pred_daily_port_calls
0 2025-11-15   port425    5           1                 24.881
1 2025-11-15   port846    5           1                 18.045
2 2025-11-15  port1429    5           1                 14.799
3 2025-11-15  port1417    5           1                 33.397
4 2025-11-15  port1416    5           1                 17.222
5 2025-11-15  port1404    5           1                 21.068
6 2025-11-15  port1378    5           1                 15.524
7 2025-11-15  port1338    5           1                 17.503
8 2025-11-15  port1308    5           1                 14.591
9 2025-11-15  port1305    5           1                 20.623


In [6]:
# =============================================================================
# Cell 6 — MERGE AND EXPORT FINAL RESULTS
# =============================================================================
pred_out = future_df[["event_date", "portid", "pred_daily_port_calls"]].copy()

try:
    existing_df = pd.read_parquet(VALIDATION_PREDS_PATH)
    # Standardize formats
    existing_df['event_date'] = pd.to_datetime(existing_df['event_date'])
    print(f"Existing predictions loaded: {len(existing_df):,} rows.")
except FileNotFoundError:
    existing_df = pd.DataFrame()
    print("No existing predictions found.")

if not existing_df.empty:
    # Merge existing (which includes historical tracking) with the newly predicted 7 days
    # Ensure we use 'pred_daily_port_calls' and align schemas
    existing_keep = existing_df[['event_date', 'portid', 'pred_daily_port_calls']].copy()
    final_df = pd.concat([existing_keep, pred_out], ignore_index=True)
    
    # Drop any possible timeline overlap duplicates safely
    final_df = final_df.drop_duplicates(subset=['portid', 'event_date'], keep='last')
else:
    final_df = pred_out

final_df = final_df.sort_values(['portid', 'event_date']).reset_index(drop=True)

final_df.to_parquet(PREDICTIONS_OUTPUT_PATH, index=False)
print(f"\u2705 Final refreshed pipeline output successfully written to: {PREDICTIONS_OUTPUT_PATH}")
print(f"  Total preserved forecasted sequence length: {len(final_df):,} rows")

Existing predictions loaded: 125,500 rows.


✅ Final refreshed pipeline output successfully written to: outputs\latest_predictions.parquet


  Total preserved forecasted sequence length: 125,850 rows
